# 🧪 Pipeline Evaluation Notebook
**Role:** Model Evaluation & Monitoring

This notebook is independent from the developer's pipeline notebook.
It loads the same models, runs them on a test audio file, and measures performance.

### ⚠️ Before running:
1. Upload `summary_evaluator.py` to Colab (Files panel → Upload)
2. Upload your test audio file (e.g. `audio_lesson.mp3`)
3. Run cells **one by one in order**

In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Install all required libraries
# ─────────────────────────────────────────────
!pip uninstall openai-whisper numba -y -q
!pip install torch==2.6.0 torchaudio==2.6.0 --break-system-packages -q
!pip install faster-whisper --break-system-packages -q
!pip install 'numpy>=1.24,<2.0' --force-reinstall --break-system-packages -q
!pip install transformers==4.46.0 sentence-transformers --break-system-packages -q
!pip install chatterbox-tts --break-system-packages -q
!pip install jiwer rouge-score bert-score pydub --break-system-packages -q
!apt-get install -y ffmpeg -q
print('✅ All libraries installed — restart runtime now if prompted')

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — Load all models
# Takes 3-5 minutes — wait for 🚀 at the end
# ─────────────────────────────────────────────
import torch
import os
os.environ['TORCHCODEC_DISABLE'] = '1'

from faster_whisper import WhisperModel
from sentence_transformers import SentenceTransformer
from chatterbox.tts import ChatterboxTTS
from transformers import BartForConditionalGeneration, BartTokenizer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'📍 Device: {DEVICE}')

print('🔊 Loading Whisper...')
whisper_model = WhisperModel('base', device=DEVICE, compute_type='int8')
print('  ✓ Whisper ready')

print('📝 Loading BART...')
bart_tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
bart_model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn').to(DEVICE)
print('  ✓ BART ready')

print('🔗 Loading SentenceTransformer...')
sim_model = SentenceTransformer('all-MiniLM-L6-v2')
print('  ✓ SentenceTransformer ready')

print('🗣️  Loading Chatterbox TTS...')
tts_model = ChatterboxTTS.from_pretrained(device=DEVICE)
print('  ✓ Chatterbox ready')

print('\n🚀 All models loaded!')

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Import the evaluator
# Make sure summary_evaluator.py is uploaded first!
# ─────────────────────────────────────────────
from summary_evaluator import (
    evaluate_asr,
    evaluate_summarization,
    evaluate_tts,
    evaluate_keywords,
    print_report,
)
import time
print('✅ Evaluator imported successfully')

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Set your reference texts (ground truth)
# These come from the BBC page — NOT from Whisper!
# ─────────────────────────────────────────────

REFERENCE_TRANSCRIPTION = """The question of why humans sleep is not easy to answer. In terms of evolution, why would it make sense to go unconscious every night, leaving yourself vulnerable to danger? It can only mean that the benefits gained from sleep are huge. If you've had a bad night's sleep and then you try and do some work, you just can't concentrate. Your brain isn't in the right state to take in new information. But we also need to sleep after we've learnt new things, because that's when our brain consolidates the information, takes it from short-term storage to long-term storage. Your brain does actually work on things during the night, and you can often wake up and have solved the problem in your sleep. Sleep deprivation is a form of stress, and when you are stressed, your fight-or-flight response is activated. If we are chronically stressed, or have chronic sleep deprivation, you might constantly have a weakened immune system, which can then lead to knock-on problems including heart disease. People who get enough sleep live about five years longer than people who don't."""

REFERENCE_SUMMARY = """Sleep is essential for human health and survival. It helps the brain consolidate new information into long-term memory and solve problems. Lack of sleep causes stress, weakens the immune system, and can lead to serious conditions like heart disease. People who sleep enough tend to live significantly longer."""

REFERENCE_KEYWORDS = ['sleep', 'brain', 'immune system', 'heart disease', 'stress', 'memory', 'evolution']

print('✅ Reference texts set')
print(f'   Transcription: {len(REFERENCE_TRANSCRIPTION.split())} words')
print(f'   Summary: {len(REFERENCE_SUMMARY.split())} words')
print(f'   Keywords: {REFERENCE_KEYWORDS}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Transcribe the audio file with Whisper
# Make sure audio_lesson.mp3 is uploaded!
# ─────────────────────────────────────────────
from pydub import AudioSegment

AUDIO_PATH = '/content/audio_lesson.mp3'  # change if your file has a different name

# Convert mp3 to wav
audio = AudioSegment.from_mp3(AUDIO_PATH)
audio.export('/content/audio_lesson.wav', format='wav')
print('✓ Converted to wav')

# Transcribe with Whisper
segments, info = whisper_model.transcribe('/content/audio_lesson.wav', beam_size=5)
transcription = ' '.join(seg.text for seg in segments).strip()

print('\n📄 WHISPER TRANSCRIPTION:')
print(transcription)
print(f'\nTotal words: {len(transcription.split())}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Summarize with BART
# ─────────────────────────────────────────────
def summarize(text, max_length=150, min_length=50):
    """Summarize text using BART. If text is short, return as-is."""
    word_count = len(text.split())
    if word_count <= max_length:
        print(f'ℹ️  Text only {word_count} words — too short to summarize, returning as-is')
        return text
    inputs = bart_tokenizer(text, return_tensors='pt', max_length=1024, truncation=True).to(DEVICE)
    summary_ids = bart_model.generate(
        inputs['input_ids'],
        max_length=max_length,
        min_length=min_length,
        length_penalty=2.0,
        num_beams=4
    )
    return bart_tokenizer.decode(summary_ids[0], skip_special_tokens=True)

summary = summarize(transcription, max_length=150)
print('\n📝 BART SUMMARY:')
print(summary)
print(f'\nSummary words: {len(summary.split())}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — Generate audio with Chatterbox TTS
# ─────────────────────────────────────────────
import torchaudio

print('🗣️  Generating TTS audio...')
t0 = time.time()
wav = tts_model.generate(summary)
generation_time = time.time() - t0

# Save and convert to bytes
torchaudio.save('/content/output_summary.wav', wav, tts_model.sr)
with open('/content/output_summary.wav', 'rb') as f:
    audio_bytes = f.read()

print(f'✓ TTS done in {generation_time:.2f}s')
print(f'  Audio saved to /content/output_summary.wav')

In [ ]:
# ─────────────────────────────────────────────
# CELL 8 — Run full evaluation & print report
# This is the final result of your work!
# ─────────────────────────────────────────────

# 1. ASR — how accurate was Whisper?
asr_metrics = evaluate_asr(
    hypothesis = transcription,
    reference  = REFERENCE_TRANSCRIPTION
)

# 2. Summarization — how good was BART?
nlp_metrics = evaluate_summarization(
    summary           = summary,
    reference_summary = REFERENCE_SUMMARY,
    original_text     = transcription,
    use_bertscore     = True
)

# 3. TTS — how fast was Chatterbox?
tts_metrics = evaluate_tts(
    audio_bytes             = audio_bytes,
    generation_time_seconds = generation_time
)

# 4. Print the full report
print_report(
    asr_metrics = asr_metrics,
    nlp_metrics = nlp_metrics,
    tts_metrics = tts_metrics
)